# CMIP6 metadata and clisops subsetting

Each synthetic NetCDF fixture is decoded with xarray and passed to one clisops request combining a time range and a geographic bounding box. The section numbers identify the QA/QC findings under review. Diagnostic wording is paraphrased. Exact reproductions match the supplied metadata value or reference on a synthetic fixture. Controls and robustness probes help interpret the results.

In [1]:
import sys
from importlib.metadata import version

import ipytest
import numpy as np
import pytest
import xarray as xr
from clisops.ops.subset import subset
from netCDF4 import Dataset

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

print(f"Python {sys.version.split()[0]}")
for package in ("clisops", "xarray", "cf-xarray", "numpy", "netCDF4", "cftime", "pytest", "ipytest"):
    print(f"{package} {version(package)}")

Python 3.13.18
clisops 0.18.1
xarray 2025.10.1
cf-xarray 0.11.3
numpy 2.5.4
netCDF4 1.7.5
cftime 1.6.5
pytest 9.2.0
ipytest 0.14.2


## Shared test data

Use four daily samples from 1 to 4 January 1850 and request the middle two days. The bbox is 12°W to 3°W and 43°N to 52°N. On the rectilinear grid it includes latitudes 45° and 50° and longitudes 10°W and 5°W. Unique data values make a shifted selection visible.

In [2]:
TIME_UNITS = "days since 1850-01-01 00:00:00"
DATES = np.arange("1850-01-01", "1850-01-05", dtype="datetime64[D]").astype("datetime64[ns]")
TIME_RANGE = "1850-01-02T00:00:00/1850-01-03T00:00:00"
BBOX = (-12, 43, -3, 52)  # west, south, east, north

time = (
    "time",
    np.arange(4),
    {"units": TIME_UNITS, "calendar": "standard", "axis": "T", "standard_name": "time"},
)
latitude = (
    "lat",
    [40, 45, 50, 55],
    {"units": "degrees_north", "axis": "Y", "standard_name": "latitude"},
)
longitude = (
    "lon",
    [-15, -10, -5, 0, 5],
    {"units": "degrees_east", "axis": "X", "standard_name": "longitude"},
)
pressure = (
    "plev",
    [100000, 85000],
    {"units": "Pa", "axis": "Z", "standard_name": "air_pressure", "positive": "down"},
)

rectilinear = xr.Dataset(
    {"ta": (("time", "plev", "lat", "lon"), np.arange(160.0).reshape(4, 2, 4, 5))},
    coords={"time": time, "plev": pressure, "lat": latitude, "lon": longitude},
)
rectilinear_bbox = {"lat": slice(1, 3), "lon": slice(1, 3)}

In [3]:
cases = []


def add_case(name, raw, variable, bbox_index, bbox_mask=None):
    # Use positional expectations, independently of clisops' coordinate discovery.
    expected = raw[variable].assign_coords(time=DATES)
    expected_subset = expected.isel(time=slice(1, 3), **bbox_index)
    if bbox_mask is not None:
        expected_subset = expected_subset.where(bbox_mask)
    cases.append(pytest.param((raw, expected, expected_subset), id=name))

## 3.1 Time-unit string formatting

*Representative diagnostic*  
The reference time in the units string uses unpadded fields.

**Exact metadata reproduction.** The variant writes `days since 1850-01-01 0:0:0.0` to NetCDF. The control uses `00:00:00`, with the same numeric offsets and standard calendar. The test checks that the unpadded string decodes to the expected dates and yields the expected time and bbox selection.

In [4]:
add_case("rectilinear-control", rectilinear, "ta", rectilinear_bbox)

short_time_units = rectilinear.copy(deep=True)
short_time_units.time.attrs["units"] = "days since 1850-01-01 0:0:0.0"
add_case("unpadded-time-units", short_time_units, "ta", rectilinear_bbox)

## 3.3 Coordinate axis / standard_name metadata

*Representative diagnostic*  
A coordinate is missing its `axis` or `standard_name` attribute.

**Reported pattern and robustness probes.** The eight variants remove one attribute at a time from `time`, `lat`, `lon`, or `plev`. Each keeps the coordinate name, units, and other attributes. These isolated removals check whether the expected time and bbox selection still works and preserves both pressure levels.

In [5]:
for coordinate in ("time", "lat", "lon", "plev"):
    for attribute in ("axis", "standard_name"):
        missing_attribute = rectilinear.copy(deep=True)
        del missing_attribute[coordinate].attrs[attribute]
        add_case(f"{coordinate}-without-{attribute}", missing_attribute, "ta", rectilinear_bbox)

## 3.7 Horizontal x/y coordinate units

*Representative diagnostic*  
Projected horizontal coordinates declare `degrees` where metre units are expected.

**Reported metadata pattern.** The variant gives `projection_x_coordinate` and `projection_y_coordinate` axes degree values and units. The metre-valued control and the variant share valid 2-D latitude and longitude. The combined test expects a 2 × 2 window with the northeast data cell masked at both selected times and the geographic coordinates preserved. The separate negative control is a robustness probe that removes both geographic coordinates.

In [6]:
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), np.arange(48.0).reshape(4, 3, 4))},
    coords={
        "time": time,
        "x": ("x", [0, 500_000, 1_000_000, 1_500_000],
              {"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"}),
        "y": ("y", [0, 500_000, 1_000_000],
              {"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"}),
        "lat": (("y", "x"), [[39, 40, 41, 42], [43, 45, 46, 47], [49, 50, 53, 54]],
                {"standard_name": "latitude", "units": "degrees_north"}),
        "lon": (("y", "x"), [[-16, -11, -6, -1], [-15, -10, -5, 0], [-14, -9, -4, 1]],
                {"standard_name": "longitude", "units": "degrees_east"}),
    },
)
curvilinear_bbox = {"y": slice(1, 3), "x": slice(1, 3)}
curvilinear_mask = xr.DataArray([[True, True], [True, False]], dims=("y", "x"))

add_case("projected-axes-in-metres", curvilinear, "tas", curvilinear_bbox, curvilinear_mask)

axes_in_degrees = curvilinear.assign_coords(
    x=("x", [0, 5, 10, 15], {**curvilinear.x.attrs, "units": "degrees"}),
    y=("y", [0, 5, 10], {**curvilinear.y.attrs, "units": "degrees"}),
)
add_case("projected-axes-in-degrees", axes_in_degrees, "tas", curvilinear_bbox, curvilinear_mask)

## 3.12 Missing auxiliary height coordinate

*Representative diagnostic*  
The data variable references `height`, which is absent.

**Exact reference reproduction.** Both files declare `coordinates="height"` on `tas`. The control has a scalar 2 m height and the variant omits it. The fixture checks that the reference survives on disk and in `tas.encoding["coordinates"]` after xarray decoding. The combined test checks whether the dangling reference affects the returned data or coordinates.

In [7]:
height_control = xr.Dataset(
    {"tas": (("time", "lat", "lon"), np.arange(80.0).reshape(4, 4, 5), {"coordinates": "height"})},
    coords={
        "time": time,
        "lat": latitude,
        "lon": longitude,
        "height": ((), 2.0, {"standard_name": "height", "units": "m", "positive": "up"}),
    },
)
add_case("height-control", height_control, "tas", rectilinear_bbox)
missing_height = height_control.drop_vars("height")
add_case("missing-height", missing_height, "tas", rectilinear_bbox)

## Pytest tests

The fixture verifies NetCDF metadata and xarray decoding before running the combined request. The negative control checks the error for a bbox request on a projected grid without geographic coordinates.

In [8]:
ipytest.clean()


@pytest.fixture(params=cases)
def decoded_case(request, tmp_path):
    raw, expected, expected_subset = request.param
    path = tmp_path / "fixture.nc"
    raw.to_netcdf(path, engine="netcdf4")

    with Dataset(path) as file:
        assert file["time"].units == raw["time"].attrs["units"]
        if raw is height_control or raw is missing_height:
            assert file[expected.name].coordinates == "height"
            assert ("height" in file.variables) == (raw is height_control)

    with xr.open_dataset(path, engine="netcdf4") as dataset:
        decoded = dataset.load()

    assert decoded[expected.name].equals(expected)
    for name, coordinate in raw.coords.items():
        for attr in ("axis", "standard_name", "units"):
            actual = decoded[name].encoding.get(attr, decoded[name].attrs.get(attr))
            assert actual == coordinate.attrs.get(attr), f"{name}.{attr}: {actual!r}"
    if raw is missing_height:
        assert decoded[expected.name].encoding.get("coordinates") == "height"
        assert "height" not in decoded.variables

    return decoded, expected_subset


def test_time_and_bbox_subset(decoded_case):
    decoded, expected_subset = decoded_case
    outputs = subset(decoded, time=TIME_RANGE, area=BBOX, output_type="xarray")
    assert len(outputs) == 1
    assert outputs[0][expected_subset.name].equals(expected_subset)


def test_bbox_without_geographic_coordinates():
    without_lat_lon = xr.decode_cf(axes_in_degrees.drop_vars(["lat", "lon"]))
    with pytest.raises(AttributeError, match="has no attribute 'lon'") as error:
        subset(without_lat_lon, area=BBOX, output_type="xarray")

    print(f"Bbox without geographic coordinates raised {error.type.__name__}: {error.value}")

## Run pytest

In [9]:
ipytest.run("-v", "-rP", "--color=no", "-p", "no:cacheprovider")

======================================= test session starts ========================================
platform darwin -- Python 3.13.18, pytest-9.2.0, pluggy-1.7.0 -- /private/tmp/c3s-cmip6-update-tests-01a1cbf2/bin/python
rootdir: /Users/pingu/Documents/GitHub/macpingu/c3s-cmip6-update-tests/notebooks
plugins: platformdirs-4.14.5, anyio-4.16.0
collecting ... collected 15 items

t_ca3bd17766304fb09351bede719bde01.py::test_time_and_bbox_subset[rectilinear-control] PASSED [  6%]
t_ca3bd17766304fb09351bede719bde01.py::test_time_and_bbox_subset[unpadded-time-units] PASSED [ 13%]
t_ca3bd17766304fb09351bede719bde01.py::test_time_and_bbox_subset[time-without-axis] PASSED   [ 20%]
t_ca3bd17766304fb09351bede719bde01.py::test_time_and_bbox_subset[time-without-standard_name] PASSED [ 26%]
t_ca3bd17766304fb09351bede719bde01.py::test_time_and_bbox_subset[lat-without-axis] PASSED    [ 33%]
t_ca3bd17766304fb09351bede719bde01.py::test_time_and_bbox_subset[lat-without-standard_name] PASSED [ 40%]
t_ca3b

<ExitCode.OK: 0>

## Results and limitations

All 14 synthetic time+bbox requests returned the expected coordinates and values.

- **3.1 Time units.** Xarray decoded `days since 1850-01-01 0:0:0.0` to the same dates as the padded control.
- **3.3 Coordinate metadata.** Removing `axis` or `standard_name` one at a time preserved the expected subset, including both pressure levels.
- **3.7 Horizontal x/y units.** Degree-valued projected axes returned the expected subset with valid 2-D latitude and longitude. The negative control without those coordinates raised an `AttributeError` for missing longitude.
- **3.12 Missing height.** Xarray retained the dangling `coordinates="height"` reference in the variable's encoding, and clisops returned the expected subset.

All 14 requests, including controls, warned that longitude cell bounds could not be identified. The tests cover isolated metadata changes in small, single-file fixtures with one calendar and a fixed time range and bbox. Compatibility across the affected real datasets remains unverified.